# 🧠 Deep Learning with PyTorch: From Tensors to Neural Networks

---

## What You'll Learn

By the end of this tutorial you will be able to:

| Section | Topics |
|---------|--------|
| **Part 1 – PyTorch Fundamentals** | Tensors, operations, broadcasting, autograd |
| **Part 2 – Your First Neural Network** | `nn.Module`, training loop, loss curves, decision boundaries |
| **Part 3 – Improving the Model** | Overfitting, dropout, batch-norm, learning-rate & optimizer comparison |
| **Part 4 – CNN Basics** | Convolutions on synthetic image data |
| **Part 5 – Model Management** | Saving / loading, next steps |

> **Prerequisites:** Basic Python and NumPy knowledge.
> All data in this notebook is **synthetic** — no downloads required.

---

# Part 1: PyTorch Fundamentals

## 1.1 What is PyTorch?

[PyTorch](https://pytorch.org/) is an open-source deep-learning framework developed by Meta AI. It provides:

* **Tensors** — GPU-accelerated multi-dimensional arrays (like NumPy `ndarray`, but with autograd).
* **Automatic differentiation (autograd)** — computes gradients through a dynamic computation graph.
* **`nn` module** — high-level building blocks for neural networks (layers, losses, optimizers).

### Why PyTorch?

| Feature | Benefit |
|---------|---------|
| Dynamic computation graphs | Easy debugging with standard Python tools |
| Pythonic API | Feels like writing plain NumPy code |
| Huge ecosystem | Hugging Face, torchvision, torchaudio, Lightning … |
| Production ready | TorchScript, ONNX export, TorchServe |

Let's start by importing the libraries we'll use throughout the notebook.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_moons, make_circles
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

print(f"PyTorch version : {torch.__version__}")
print(f"NumPy version   : {np.__version__}")

## 1.2 Tensor Basics

A **tensor** is the fundamental data structure in PyTorch — an $n$-dimensional array that can live on CPU or GPU and track gradients.

### Creating tensors

| Function | Description |
|----------|-------------|
| `torch.tensor(data)` | From a Python list / NumPy array |
| `torch.zeros(shape)` | All zeros |
| `torch.ones(shape)` | All ones |
| `torch.randn(shape)` | Standard-normal random values |
| `torch.from_numpy(arr)` | Zero-copy view of a NumPy array |

In [ ]:
# From a Python list
a = torch.tensor([1.0, 2.0, 3.0])
print("From list      :", a, "| dtype:", a.dtype)

# Zeros and ones
z = torch.zeros(2, 3)
o = torch.ones(2, 3, dtype=torch.int32)
print("Zeros (2x3)    :\n", z)
print("Ones  (2x3 int):\n", o)

# Random normal
r = torch.randn(3, 4)
print("Randn (3x4)    :\n", r)

# From NumPy (shares memory)
np_arr = np.array([4.0, 5.0, 6.0])
t_from_np = torch.from_numpy(np_arr)
print("From NumPy     :", t_from_np)

# Shape, dtype, device
print(f"\nShape: {r.shape}  |  dtype: {r.dtype}  |  device: {r.device}")

## 1.3 Tensor Operations

PyTorch supports element-wise arithmetic, matrix multiplication, reshaping, and advanced indexing — all with familiar syntax.

> **Tip:** The `@` operator and `torch.mm` both perform matrix multiplication.

In [ ]:
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
y = torch.tensor([[5.0, 6.0], [7.0, 8.0]])

# Element-wise arithmetic
print("Add :\n", x + y)
print("Mul :\n", x * y)

# Matrix multiplication (three equivalent ways)
print("MatMul (@)         :\n", x @ y)
print("MatMul (torch.mm)  :\n", torch.mm(x, y))
print("MatMul (torch.matmul):", torch.matmul(x, y))

# Reshaping
flat = torch.arange(12)
print("\nOriginal :", flat)
print("view(3,4):\n", flat.view(3, 4))
print("reshape(2,6):\n", flat.reshape(2, 6))

# Indexing & slicing
m = torch.randn(4, 5)
print("\nRow 0        :", m[0])
print("Col 2        :", m[:, 2])
print("Boolean mask :", m[m > 0.5])

## 1.4 Autograd — Automatic Differentiation

PyTorch builds a **dynamic computation graph** on the fly. When you call `.backward()` on a scalar, it computes $\partial\text{output}/\partial\text{leaf}$ for every leaf tensor that has `requires_grad=True`.

### Simple example

For $y = 3x^2 + 2x + 1$ at $x=2$:

$$\frac{dy}{dx} = 6x + 2 = 14$$

In [ ]:
# Create a leaf tensor with gradient tracking
x = torch.tensor(2.0, requires_grad=True)

# Forward pass
y = 3 * x ** 2 + 2 * x + 1  # y = 3(4) + 4 + 1 = 17

# Backward pass
y.backward()

print(f"x     = {x.item()}")
print(f"y     = {y.item()}")
print(f"dy/dx = {x.grad.item()}  (expected 14)")

In [ ]:
# Autograd with vectors — Jacobian-vector product example
x = torch.randn(3, requires_grad=True)
y = x * 2
z = y.sum()  # scalar output so we can call .backward()
z.backward()

print(f"x      = {x.data}")
print(f"dz/dx  = {x.grad}")  # should be [2, 2, 2]

## 1.5 GPU Awareness

PyTorch can transparently move computations to a CUDA GPU when one is available. The idiomatic pattern is:

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tensor = tensor.to(device)
model  = model.to(device)
```

> **Note:** This tutorial runs on **CPU** — the code still works identically; tensors simply stay on `"cpu"`.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Using device  : {device}")

# Move a tensor to the active device
t = torch.randn(3, 3).to(device)
print(f"Tensor device : {t.device}")

---

# Part 2: Building Your First Neural Network

We'll classify points from scikit-learn's **`make_moons`** dataset — a classic non-linearly-separable toy problem.

## 2.1 The Dataset — Make Moons

`make_moons` generates two interleaving half-circles. It's simple enough to visualize in 2-D yet requires a non-linear decision boundary, making it perfect for demonstrating neural networks.

In [ ]:
# Generate data
X_np, y_np = make_moons(n_samples=1000, noise=0.2, random_state=42)

# Train / test split
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X_np, y_np, test_size=0.2, random_state=42
)

# Convert to PyTorch tensors
X_train = torch.from_numpy(X_train_np).float().to(device)
y_train = torch.from_numpy(y_train_np).float().unsqueeze(1).to(device)
X_test  = torch.from_numpy(X_test_np).float().to(device)
y_test  = torch.from_numpy(y_test_np).float().unsqueeze(1).to(device)

print(f"X_train shape: {X_train.shape}  y_train shape: {y_train.shape}")
print(f"X_test  shape: {X_test.shape}   y_test  shape: {y_test.shape}")

# Visualize
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(X_np[:, 0], X_np[:, 1], c=y_np, cmap="coolwarm", s=15, alpha=0.8)
ax.set_title("Make Moons Dataset")
ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
plt.colorbar(scatter, ax=ax, label="Class")
plt.tight_layout()
plt.show()

## 2.2 The `nn.Module` Pattern

Every PyTorch model inherits from `nn.Module`. You need to:

1. **`__init__`** — define the layers.
2. **`forward`** — describe how data flows through the layers.

PyTorch handles the backward pass automatically via autograd.

```
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer = nn.Linear(in_features, out_features)

    def forward(self, x):
        return self.layer(x)
```

## 2.3 Building an MLP Classifier

Our network has three linear layers with ReLU activations and a final Sigmoid to output a probability between 0 and 1.

```
Input (2) → Linear(32) → ReLU → Linear(16) → ReLU → Linear(1) → Sigmoid
```

In [ ]:
class MoonClassifier(nn.Module):
    # Simple MLP for binary classification on 2-D data.

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)


model = MoonClassifier().to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## 2.4 Loss Function & Optimizer

| Component | Choice | Why |
|-----------|--------|-----|
| **Loss** | `nn.BCELoss` | Binary cross-entropy — standard for binary classification with sigmoid output |
| **Optimizer** | `Adam` | Adaptive learning rate — works well out of the box |
| **Learning rate** | `0.01` | A common starting point |

> **Learning rate** controls how large each parameter update is. Too high → divergence; too low → slow convergence.

In [ ]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

## 2.5 The Training Loop

The canonical PyTorch training loop:

```
for epoch in range(n_epochs):
    y_pred = model(X)            # 1. Forward pass
    loss   = criterion(y_pred, y) # 2. Compute loss
    optimizer.zero_grad()         # 3. Zero gradients
    loss.backward()               # 4. Backward pass
    optimizer.step()              # 5. Update weights
```

In [ ]:
torch.manual_seed(42)
model = MoonClassifier().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.BCELoss()

epochs = 200
loss_history = []

for epoch in range(epochs):
    # Forward
    y_pred = model(X_train)
    loss = criterion(y_pred, y_train)

    # Backward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    loss_history.append(loss.item())
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:>4d}/{epochs}  Loss: {loss.item():.4f}")

## 2.6 Training Loss Curve

A decreasing loss curve tells us the model is learning. Plateaus may indicate we need a different learning rate or architecture.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(loss_history, linewidth=1.5)
ax.set_xlabel("Epoch")
ax.set_ylabel("BCE Loss")
ax.set_title("Training Loss Curve")
plt.tight_layout()
plt.show()

## 2.7 Evaluation & Decision Boundary

We evaluate the model on the held-out test set and visualize the learned decision boundary using a meshgrid.

In [ ]:
# Accuracy helper
def compute_accuracy(model, X, y):
    with torch.no_grad():
        probs = model(X)
        preds = (probs >= 0.5).float()
    return (preds == y).float().mean().item()

train_acc = compute_accuracy(model, X_train, y_train)
test_acc  = compute_accuracy(model, X_test, y_test)
print(f"Train accuracy: {train_acc:.4f}")
print(f"Test  accuracy: {test_acc:.4f}")

In [ ]:
def plot_decision_boundary(model, X, y, title="Decision Boundary"):
    # Plot the model decision boundary over the 2-D input space.
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                         np.linspace(y_min, y_max, 300))
    grid = torch.from_numpy(np.c_[xx.ravel(), yy.ravel()]).float().to(device)

    with torch.no_grad():
        zz = model(grid).cpu().numpy().reshape(xx.shape)

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.contourf(xx, yy, zz, levels=50, cmap="RdBu", alpha=0.7)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=15, edgecolors="k", linewidths=0.3)
    ax.set_title(title)
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    plt.tight_layout()
    plt.show()


plot_decision_boundary(model, X_test_np, y_test_np, title="MLP Decision Boundary (Test Set)")

---

# Part 3: Improving the Model

In this section we'll explore **overfitting**, **regularization**, **learning-rate tuning**, and **optimizer comparison**.

## 3.1 Overfitting Demo

An **over-parameterized** model can memorize the training data and perform poorly on unseen data. Let's demonstrate this with a very large network trained on a small subset of the data.

In [ ]:
# Small dataset to make overfitting easy
X_small, y_small = make_moons(n_samples=100, noise=0.2, random_state=42)
X_sm_train, X_sm_test, y_sm_train, y_sm_test = train_test_split(
    X_small, y_small, test_size=0.3, random_state=42
)
X_sm_train_t = torch.from_numpy(X_sm_train).float().to(device)
y_sm_train_t = torch.from_numpy(y_sm_train).float().unsqueeze(1).to(device)
X_sm_test_t  = torch.from_numpy(X_sm_test).float().to(device)
y_sm_test_t  = torch.from_numpy(y_sm_test).float().unsqueeze(1).to(device)


class OverfitNet(nn.Module):
    # Deliberately over-sized network.
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)


torch.manual_seed(42)
big_model = OverfitNet().to(device)
opt_big = optim.Adam(big_model.parameters(), lr=0.005)
crit = nn.BCELoss()

train_losses, test_losses = [], []
for epoch in range(500):
    # Train
    big_model.train()
    pred = big_model(X_sm_train_t)
    loss = crit(pred, y_sm_train_t)
    opt_big.zero_grad(); loss.backward(); opt_big.step()
    train_losses.append(loss.item())

    # Eval
    big_model.eval()
    with torch.no_grad():
        test_losses.append(crit(big_model(X_sm_test_t), y_sm_test_t).item())

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(train_losses, label="Train Loss")
ax.plot(test_losses, label="Test Loss")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.set_title("Overfitting: Train vs Test Loss")
ax.legend()
plt.tight_layout(); plt.show()

print(f"Train acc: {compute_accuracy(big_model, X_sm_train_t, y_sm_train_t):.4f}")
print(f"Test  acc: {compute_accuracy(big_model, X_sm_test_t, y_sm_test_t):.4f}")

## 3.2 Regularization: Dropout & Batch Normalization

Two popular techniques to combat overfitting:

| Technique | How it works |
|-----------|--------------|
| **Dropout** | Randomly sets a fraction of activations to 0 during training, preventing co-adaptation |
| **Batch Normalization** | Normalizes activations within a mini-batch, stabilizing and accelerating training |

In [ ]:
class RegularizedNet(nn.Module):
    # MLP with BatchNorm and Dropout.
    def __init__(self, dropout_rate=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)


torch.manual_seed(42)
reg_model = RegularizedNet(dropout_rate=0.3).to(device)
opt_reg = optim.Adam(reg_model.parameters(), lr=0.005)

reg_train_losses, reg_test_losses = [], []
for epoch in range(500):
    reg_model.train()
    pred = reg_model(X_sm_train_t)
    loss = crit(pred, y_sm_train_t)
    opt_reg.zero_grad(); loss.backward(); opt_reg.step()
    reg_train_losses.append(loss.item())

    reg_model.eval()
    with torch.no_grad():
        reg_test_losses.append(crit(reg_model(X_sm_test_t), y_sm_test_t).item())

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
axes[0].plot(train_losses, label="Train"); axes[0].plot(test_losses, label="Test")
axes[0].set_title("Without Regularization"); axes[0].legend(); axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[1].plot(reg_train_losses, label="Train"); axes[1].plot(reg_test_losses, label="Test")
axes[1].set_title("With Dropout + BatchNorm"); axes[1].legend(); axes[1].set_xlabel("Epoch")
plt.suptitle("Effect of Regularization", fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

print(f"Regularized model — Train acc: {compute_accuracy(reg_model, X_sm_train_t, y_sm_train_t):.4f}  "
      f"Test acc: {compute_accuracy(reg_model, X_sm_test_t, y_sm_test_t):.4f}")

## 3.3 Learning Rate Comparison

The learning rate is arguably the **most important hyperparameter**. Let's train the same architecture with three different learning rates and compare loss curves.

In [ ]:
learning_rates = [0.1, 0.01, 0.001]
lr_histories = {}

for lr in learning_rates:
    torch.manual_seed(42)
    m = MoonClassifier().to(device)
    opt = optim.Adam(m.parameters(), lr=lr)
    losses = []
    for _ in range(300):
        pred = m(X_train)
        loss = criterion(pred, y_train)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    lr_histories[lr] = losses

fig, ax = plt.subplots(figsize=(7, 4))
for lr, losses in lr_histories.items():
    ax.plot(losses, label=f"lr={lr}")
ax.set_xlabel("Epoch"); ax.set_ylabel("BCE Loss")
ax.set_title("Effect of Learning Rate")
ax.legend(); plt.tight_layout(); plt.show()

## 3.4 Optimizer Comparison: SGD vs Adam vs AdamW

| Optimizer | Key Idea |
|-----------|----------|
| **SGD** | Vanilla gradient descent (+ optional momentum) |
| **Adam** | Adaptive per-parameter learning rates with momentum |
| **AdamW** | Adam with decoupled weight decay (better generalization) |

In [ ]:
optimizer_configs = {
    "SGD (lr=0.1, momentum=0.9)": lambda p: optim.SGD(p, lr=0.1, momentum=0.9),
    "Adam (lr=0.01)": lambda p: optim.Adam(p, lr=0.01),
    "AdamW (lr=0.01, wd=1e-2)": lambda p: optim.AdamW(p, lr=0.01, weight_decay=1e-2),
}

opt_histories = {}
for name, opt_fn in optimizer_configs.items():
    torch.manual_seed(42)
    m = MoonClassifier().to(device)
    opt = opt_fn(m.parameters())
    losses = []
    for _ in range(300):
        pred = m(X_train)
        loss = criterion(pred, y_train)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    opt_histories[name] = losses

fig, ax = plt.subplots(figsize=(7, 4))
for name, losses in opt_histories.items():
    ax.plot(losses, label=name)
ax.set_xlabel("Epoch"); ax.set_ylabel("BCE Loss")
ax.set_title("Optimizer Comparison")
ax.legend(); plt.tight_layout(); plt.show()

---

# Part 4: CNN Basics (Bonus)

## 4.1 What Are Convolutional Neural Networks?

A **CNN** applies learned spatial filters (kernels) that slide across an image to produce **feature maps**.

### Key building blocks

| Layer | Purpose |
|-------|---------|
| `nn.Conv2d` | Applies 2-D convolution filters |
| `nn.MaxPool2d` | Down-samples by taking the max in a window |
| `nn.Flatten` | Flattens feature maps into a 1-D vector for the classifier |
| `nn.Linear` | Fully-connected classifier head |

### Why CNNs?

* **Parameter sharing** — the same kernel is reused across the image.
* **Translation equivariance** — a feature detected at one location is detected everywhere.

## 4.2 Synthetic Image Data

We'll create simple **16×16** binary images:
- **Class 0:** horizontal lines
- **Class 1:** vertical lines

These are easy to visualize and sufficient to demonstrate convolution.

In [ ]:
def make_line_images(n_samples=500, img_size=16):
    # Generate synthetic images with horizontal (0) or vertical (1) lines.
    images, labels = [], []
    for _ in range(n_samples):
        img = np.zeros((img_size, img_size), dtype=np.float32)
        if np.random.rand() > 0.5:
            # Vertical lines
            cols = np.random.choice(img_size, size=np.random.randint(2, 5), replace=False)
            img[:, cols] = 1.0
            labels.append(1)
        else:
            # Horizontal lines
            rows = np.random.choice(img_size, size=np.random.randint(2, 5), replace=False)
            img[rows, :] = 1.0
            labels.append(0)
        # Add a little noise
        img += np.random.randn(img_size, img_size).astype(np.float32) * 0.1
        images.append(img)
    return np.array(images), np.array(labels)


np.random.seed(42)
imgs, lbls = make_line_images(n_samples=800, img_size=16)

# Show samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes[0]):
    idx = np.where(lbls == 0)[0][i]
    ax.imshow(imgs[idx], cmap="gray"); ax.set_title("Horizontal"); ax.axis("off")
for i, ax in enumerate(axes[1]):
    idx = np.where(lbls == 1)[0][i]
    ax.imshow(imgs[idx], cmap="gray"); ax.set_title("Vertical"); ax.axis("off")
plt.suptitle("Synthetic Line Images (16×16)", fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
# Prepare tensors — Conv2d expects (N, C, H, W)
X_img = torch.from_numpy(imgs).unsqueeze(1)  # add channel dim
y_img = torch.from_numpy(lbls).float().unsqueeze(1)

X_img_train, X_img_test, y_img_train, y_img_test = train_test_split(
    X_img, y_img, test_size=0.2, random_state=42
)
X_img_train = X_img_train.to(device)
y_img_train = y_img_train.to(device)
X_img_test  = X_img_test.to(device)
y_img_test  = y_img_test.to(device)

print(f"Train images: {X_img_train.shape}  Labels: {y_img_train.shape}")
print(f"Test  images: {X_img_test.shape}   Labels: {y_img_test.shape}")

## 4.3 Building a Simple CNN

Our architecture:

```
Conv2d(1→8, 3×3) → ReLU → MaxPool(2) → Conv2d(8→16, 3×3) → ReLU → MaxPool(2) → Flatten → Linear → Sigmoid
```

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),   # (N,8,16,16)
            nn.ReLU(),
            nn.MaxPool2d(2),                              # (N,8,8,8)
            nn.Conv2d(8, 16, kernel_size=3, padding=1),  # (N,16,8,8)
            nn.ReLU(),
            nn.MaxPool2d(2),                              # (N,16,4,4)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),                                 # (N,256)
            nn.Linear(16 * 4 * 4, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


cnn = SimpleCNN().to(device)
print(cnn)
print(f"\nTotal parameters: {sum(p.numel() for p in cnn.parameters()):,}")

## 4.4 Training and Evaluating the CNN

In [ ]:
torch.manual_seed(42)
cnn = SimpleCNN().to(device)
cnn_optimizer = optim.Adam(cnn.parameters(), lr=0.001)
cnn_criterion = nn.BCELoss()

cnn_losses = []
for epoch in range(80):
    cnn.train()
    pred = cnn(X_img_train)
    loss = cnn_criterion(pred, y_img_train)
    cnn_optimizer.zero_grad(); loss.backward(); cnn_optimizer.step()
    cnn_losses.append(loss.item())
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:>3d}/80  Loss: {loss.item():.4f}")

# Loss curve
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(cnn_losses, linewidth=1.5)
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.set_title("CNN Training Loss")
plt.tight_layout(); plt.show()

# Accuracy
cnn.eval()
with torch.no_grad():
    train_preds = (cnn(X_img_train) >= 0.5).float()
    test_preds  = (cnn(X_img_test)  >= 0.5).float()
train_acc = (train_preds == y_img_train).float().mean().item()
test_acc  = (test_preds  == y_img_test).float().mean().item()
print(f"\nCNN Train accuracy: {train_acc:.4f}")
print(f"CNN Test  accuracy: {test_acc:.4f}")

In [ ]:
# Show some CNN predictions
cnn.eval()
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
label_map = {0: "Horizontal", 1: "Vertical"}
for i, ax in enumerate(axes.flat):
    img = X_img_test[i].cpu()
    with torch.no_grad():
        pred_prob = cnn(X_img_test[i:i+1]).item()
    pred_label = 1 if pred_prob >= 0.5 else 0
    true_label = int(y_img_test[i].item())
    color = "green" if pred_label == true_label else "red"
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(f"P:{label_map[pred_label]}\nT:{label_map[true_label]}", color=color, fontsize=9)
    ax.axis("off")
plt.suptitle("CNN Predictions (Green=Correct, Red=Wrong)", fontsize=13)
plt.tight_layout(); plt.show()

---

# Part 5: Model Management

## 5.1 Saving & Loading Models

PyTorch offers two approaches:

| Method | Saves | Recommended? |
|--------|-------|--------------|
| `torch.save(model.state_dict(), path)` | Weights only | ✅ Yes |
| `torch.save(model, path)` | Entire model (pickle) | ⚠️ Fragile |

Always prefer saving the **state dict** — it's more portable and avoids pickle pitfalls.

In [ ]:
import os

# Save to the same directory as this notebook
save_path = os.path.join(".", "moon_classifier.pth")
torch.save(model.state_dict(), save_path)
print(f"Model saved to {save_path}")

# Load into a fresh model
loaded_model = MoonClassifier().to(device)
loaded_model.load_state_dict(torch.load(save_path, weights_only=True))
loaded_model.eval()

# Verify identical predictions
with torch.no_grad():
    original_preds = model(X_test)
    loaded_preds   = loaded_model(X_test)
print(f"Predictions match: {torch.allclose(original_preds, loaded_preds)}")

## 5.2 Summary — What We Learned

🎉 **Congratulations!** You've covered the core concepts of deep learning with PyTorch:

| Concept | Key Takeaway |
|---------|-------------|
| **Tensors** | GPU-ready arrays with autograd support |
| **Autograd** | Automatic gradient computation via dynamic graphs |
| **`nn.Module`** | The base class for all models: define `__init__` + `forward` |
| **Training loop** | Forward → loss → zero_grad → backward → step |
| **Regularization** | Dropout + BatchNorm reduce overfitting |
| **Hyperparameters** | Learning rate and optimizer choice matter a lot |
| **CNNs** | Convolutions extract spatial features from images |
| **Model I/O** | Always save `state_dict()`, not the full model |

## 5.3 🏋️ Exercise: Beat the Baseline

**Challenge:** Modify the `MoonClassifier` architecture and/or training setup to achieve **> 98 % test accuracy** on the full `make_moons` dataset (1 000 samples, 20 % noise).

Ideas to try:

1. Add more layers or wider layers.
2. Experiment with different activation functions (`nn.LeakyReLU`, `nn.GELU`).
3. Use `nn.BatchNorm1d` between layers.
4. Try `make_circles` instead — can the same architecture classify circles?
5. Implement a **learning-rate scheduler** (`torch.optim.lr_scheduler.StepLR`).

```python
# Starter code
class ImprovedClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        # YOUR ARCHITECTURE HERE
        ...

    def forward(self, x):
        ...
```

## 5.4 Next Steps

Now that you understand the fundamentals, here are some directions to explore:

| Area | Resources |
|------|-----------|
| **Computer Vision** | [torchvision](https://pytorch.org/vision/) — image classification, detection, segmentation |
| **Natural Language Processing** | [Hugging Face Transformers](https://huggingface.co/docs/transformers/) — BERT, GPT, etc. |
| **PyTorch Lightning** | [lightning.ai](https://lightning.ai/) — reduces boilerplate, adds logging & multi-GPU |
| **Experiment Tracking** | [Weights & Biases](https://wandb.ai/), [MLflow](https://mlflow.org/) |
| **Deployment** | TorchScript, ONNX, TorchServe |

> **Tip:** The official [PyTorch tutorials](https://pytorch.org/tutorials/) are an excellent next step.

---

*Happy learning! 🚀*